# PadelPro Vision — gerar CSV com o `padel_analytics` (Colab)

Corre o pipeline do João Silva numa GPU do Colab e exporta o `data.csv`
(posições dos jogadores em metros) para depois analisares no teu repo `padelpro-vision`.

**Antes de começar:** menu `Runtime → Change runtime type → T4 GPU`.

Fluxo: GPU → clonar → instalar → pesos → escolher keypoints do campo → configurar → correr → descarregar CSV.

## 1. Confirmar GPU

In [ ]:
!nvidia-smi

## 2. Clonar o repo do João

In [ ]:
%cd /content
![ -d padel_analytics ] || git clone https://github.com/Joao-M-Silva/padel_analytics.git
%cd /content/padel_analytics
!ls

## 3. Instalar dependências
Instalamos uma lista curada (o `requirements.txt` original tem um erro de escrita — `kaleav` — que parte o pip). O PyTorch já vem no Colab.

In [ ]:
!pip install -q ultralytics supervision opencv-python-headless pims plotly seaborn parse matplotlib gdown
print('Dependências instaladas.')

## 4. Descarregar os pesos
Os pesos estão na Drive do João. Descarregamos a pasta e organizamos na estrutura que o `config.py` espera:
```
weights/players_detection/yolov8m.pt
weights/players_keypoints_detection/best.pt
weights/ball_detection/TrackNet_best.pt
weights/ball_detection/InpaintNet_best.pt
weights/court_keypoints_detection/best.pt
```

In [ ]:
import gdown, os
# Pasta de pesos partilhada no README do repo
FOLDER = 'https://drive.google.com/drive/folders/1joO7w1Am7B418SIqGBq90YipQl81FMzh'
gdown.download_folder(FOLDER, output='/content/padel_analytics/_weights_dl', quiet=False, use_cookies=False)
print('\n--- ficheiros descarregados ---')
for root, _, files in os.walk('/content/padel_analytics/_weights_dl'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# Arruma os ficheiros .pt na estrutura esperada (best-effort por nome).
import os, shutil, glob
DST = {
    'yolov8m.pt': 'weights/players_detection/yolov8m.pt',
    'TrackNet_best.pt': 'weights/ball_detection/TrackNet_best.pt',
    'InpaintNet_best.pt': 'weights/ball_detection/InpaintNet_best.pt',
}
found = {os.path.basename(p): p for p in glob.glob('/content/padel_analytics/_weights_dl/**/*.pt', recursive=True)}
for name, dst in DST.items():
    if name in found:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(found[name], dst); print('OK ->', dst)
    else:
        print('FALTA:', name, '(verifica a lista acima e copia à mão)')
print('\nNOTA: há dois ficheiros best.pt (pose dos jogadores e keypoints do campo).')
print('Confirma na lista acima qual é qual e copia para:')
print('  weights/players_keypoints_detection/best.pt')
print('  weights/court_keypoints_detection/best.pt')

## 5. Escolher o vídeo
Para um primeiro teste usa o clip que vem no repo (`examples/videos/rally.mp4`).
Para usar um vídeo teu, descomenta o bloco de upload.

In [ ]:
VIDEO_PATH = '/content/padel_analytics/examples/videos/rally.mp4'

# --- para usar o teu próprio clip, descomenta: ---
# from google.colab import files
# up = files.upload()
# VIDEO_PATH = '/content/padel_analytics/' + list(up.keys())[0]

print('Vídeo:', VIDEO_PATH)

## 6. Selecionar os 12 keypoints do campo
O Colab não consegue abrir a janela interativa do repo, por isso fazemos manualmente (uma vez por câmara/court).

A célula mostra o 1.º frame. **Passa o rato** por cima de cada keypoint e lê as coordenadas `x, y` na tooltip.
Segue a ordem do diagrama (k1 a k12):
```
        k11--------------------k12
        |                       |
        k8-----------k9--------k10
        |            |          |
        k6----------------------k7
        |            |          |
        k3-----------k4---------k5
        |                       |
        k1----------------------k2
```

In [ ]:
import cv2, plotly.express as px
cap = cv2.VideoCapture(VIDEO_PATH); ok, frame = cap.read(); cap.release()
assert ok, 'Não consegui ler o vídeo'
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
fig = px.imshow(frame_rgb)
fig.update_layout(height=600, title='Passa o rato nos keypoints e lê x,y')
fig.show()

In [ ]:
import json, os
# Preenche os 12 pares (x, y) pela ordem k1..k12, lidos da imagem acima.
SELECTED_KEYPOINTS = [
    # (x1, y1),   # k1
    # (x2, y2),   # k2
    # ... até k12
]
assert len(SELECTED_KEYPOINTS) == 12, 'Precisas de exatamente 12 keypoints (k1..k12).'
os.makedirs('/content/padel_analytics/cache', exist_ok=True)
with open('/content/padel_analytics/cache/fixed_keypoints_detection.json', 'w') as f:
    json.dump([[float(x), float(y)] for x, y in SELECTED_KEYPOINTS], f)
print('Keypoints guardados em cache/fixed_keypoints_detection.json')

## 7. Configurar o `config.py`
Reescrevemos o `config.py` para: usar o vídeo escolhido, carregar os keypoints do JSON (sem janela),
limitar a poucos frames no primeiro teste, ativar a recolha de dados e usar cache (`LOAD=None` no 1.º run).

In [ ]:
MAX_FRAMES = 300   # ~10s a 30fps para o 1.º teste; põe None para o vídeo todo
config = f'''""" Config gerado pelo notebook PadelPro Vision """
INPUT_VIDEO_PATH = "{VIDEO_PATH}"
OUTPUT_VIDEO_PATH = "results.mp4"
COLLECT_DATA = True
COLLECT_DATA_PATH = "data.csv"
MAX_FRAMES = {MAX_FRAMES}

FIXED_COURT_KEYPOINTS_LOAD_PATH = "./cache/fixed_keypoints_detection.json"
FIXED_COURT_KEYPOINTS_SAVE_PATH = None

PLAYERS_TRACKER_MODEL = "./weights/players_detection/yolov8m.pt"
PLAYERS_TRACKER_BATCH_SIZE = 8
PLAYERS_TRACKER_ANNOTATOR = "rectangle_bounding_box"
PLAYERS_TRACKER_LOAD_PATH = None
PLAYERS_TRACKER_SAVE_PATH = "./cache/players_detections.json"

PLAYERS_KEYPOINTS_TRACKER_MODEL = "./weights/players_keypoints_detection/best.pt"
PLAYERS_KEYPOINTS_TRACKER_TRAIN_IMAGE_SIZE = 1280
PLAYERS_KEYPOINTS_TRACKER_BATCH_SIZE = 8
PLAYERS_KEYPOINTS_TRACKER_LOAD_PATH = None
PLAYERS_KEYPOINTS_TRACKER_SAVE_PATH = "./cache/players_keypoints_detections.json"

BALL_TRACKER_MODEL = "./weights/ball_detection/TrackNet_best.pt"
BALL_TRACKER_INPAINT_MODEL = "./weights/ball_detection/InpaintNet_best.pt"
BALL_TRACKER_BATCH_SIZE = 8
BALL_TRACKER_MEDIAN_MAX_SAMPLE_NUM = 400
BALL_TRACKER_LOAD_PATH = None
BALL_TRACKER_SAVE_PATH = "./cache/ball_detections.json"

KEYPOINTS_TRACKER_MODEL = "./weights/court_keypoints_detection/best.pt"
KEYPOINTS_TRACKER_BATCH_SIZE = 8
KEYPOINTS_TRACKER_MODEL_TYPE = "yolo"
KEYPOINTS_TRACKER_LOAD_PATH = None
KEYPOINTS_TRACKER_SAVE_PATH = None
'''
with open('/content/padel_analytics/config.py', 'w') as f:
    f.write(config)
print('config.py escrito.')

## 8. Correr o pipeline
Gera o `data.csv`. No 1.º teste pode demorar alguns minutos (faz inferência dos 4 modelos).

In [ ]:
%cd /content/padel_analytics
!python main.py

## 9. Ver e descarregar o CSV

In [ ]:
import pandas as pd
df = pd.read_csv('/content/padel_analytics/data.csv')
print('linhas:', len(df), '| colunas:', len(df.columns))
print([c for c in df.columns if c.startswith('player') and ('_x' in c or '_y' in c or 'Vnorm1' in c)][:12])
df.head()

In [ ]:
from google.colab import files
files.download('/content/padel_analytics/data.csv')
# (opcional) descarregar também as deteções da bola para o módulo de pancadas/tempo útil:
# files.download('/content/padel_analytics/cache/ball_detections.json')

## 10. Próximo passo (no teu Mac)
Com o `data.csv` descarregado, no repo `padelpro-vision`:
```bash
python -m padelpro.cli analyze --players data.csv --fps 30 --out analysis.json
```
Ajusta os nomes das colunas em `config.yaml` se necessário.